<a href="https://colab.research.google.com/github/VictorNevola/ml-study/blob/main/ml_04_treino_teste_validacao_cruzada_grid_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

df = pd.read_csv("housing.csv")
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)
print(f"R² on the test set: {tree.score(X_test, y_test):.3f}")

R² on the test set: 0.665


In [2]:
for rs in [0, 1, 7, 42, 100]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=rs)
    m = DecisionTreeRegressor(random_state=42).fit(Xtr, ytr)
    print(f"random_state={rs:3d} -> R² = {m.score(Xte, yte):.3f}")

random_state=  0 -> R² = 0.655
random_state=  1 -> R² = 0.633
random_state=  7 -> R² = 0.658
random_state= 42 -> R² = 0.665
random_state=100 -> R² = 0.651


In [3]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    DecisionTreeRegressor(random_state=42),
    X_train, y_train,          # only the training data!
    cv=5, scoring="r2"
)

print("Score of each fold:", np.round(scores, 3))
print(f"Mean: {scores.mean():.3f}  (+/- {scores.std():.3f})")

Score of each fold: [0.658 0.585 0.656 0.641 0.638]
Mean: 0.636  (+/- 0.026)


In [4]:
tree_full = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)
print(f"R² on the training data: {tree_full.score(X_train, y_train):.3f}")
print(f"R² in cross-validation:  {scores.mean():.3f}")

R² on the training data: 1.000
R² in cross-validation:  0.636


In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [3, 5, 8, 10, 12, None],
    "min_samples_leaf": [1, 5, 20, 50],
}

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5, scoring="r2"
)
grid.fit(X_train, y_train)     # tries every combination, with cross-validation

print("Best parameters:", grid.best_params_)
print(f"Best cross-validation R²: {grid.best_score_:.3f}")

Best parameters: {'max_depth': None, 'min_samples_leaf': 20}
Best cross-validation R²: 0.743


In [6]:
best_tree = grid.best_estimator_

print("Tuned tree:")
print(f"  R² on training: {best_tree.score(X_train, y_train):.3f}")
print(f"  R² on test:     {best_tree.score(X_test, y_test):.3f}")
print()
print("Original (default) tree for comparison:")
print(f"  R² on training: {tree_full.score(X_train, y_train):.3f}")
print(f"  R² on test:     {tree_full.score(X_test, y_test):.3f}")

Tuned tree:
  R² on training: 0.818
  R² on test:     0.749

Original (default) tree for comparison:
  R² on training: 1.000
  R² on test:     0.665
